In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import ast
import matplotlib.pyplot as plt

# Dizin yolları
DATA_DIR = Path("../data/raw")
LABELS_PATH = DATA_DIR / "labeled_anomalies.csv"

# Etiketleri oku
df_labels = pd.read_csv(LABELS_PATH)

def run_zscore_baseline(chan_id: str, threshold: float = 3.0):
    # 1. Yalnızca Train verisini oku
    train_data = np.load(DATA_DIR / "train" / f"{chan_id}.npy")
    train_telemetry = train_data[:, 0]
    
    # 2. İstatistikleri YALNIZCA Train verisinden hesapla (Data Leakage Önlemi)
    mu = np.mean(train_telemetry)
    sigma = np.std(train_telemetry)
    
    # Sıfıra bölme hatasını önle
    if sigma == 0:
        sigma = 1e-6
        
    # 3. Test verisini oku
    test_data = np.load(DATA_DIR / "test" / f"{chan_id}.npy")
    test_telemetry = test_data[:, 0]
    
    # 4. Test sinyali için Z-score hesapla
    z_scores = np.abs((test_telemetry - mu) / sigma)
    
    # Eşiği aşan noktaları aday anomali olarak işaretle (Boolean maske)
    anomaly_predictions = z_scores > threshold
    
    print(f"Kanal: {chan_id}")
    print(f"Train Ortalama (mu): {mu:.4f} | Train Std (sigma): {sigma:.4f}")
    print(f"Test Zaman Adımı: {len(test_telemetry)}")
    print(f"Z-score > {threshold} ile Anomali İşaretlenen Adım Sayısı: {np.sum(anomaly_predictions)}")
    
    return test_telemetry, z_scores, anomaly_predictions

# Az önce incelediğimiz E-2 (Point) ve P-1 (Contextual) kanallarını test edelim
print("--- E-2 (Point Anomaly Kanalı) ---")
_, _, pred_e2 = run_zscore_baseline("E-2", threshold=3.0)

print("\n--- P-1 (Contextual Anomaly Kanalı) ---")
_, _, pred_p1 = run_zscore_baseline("P-1", threshold=3.0)

--- E-2 (Point Anomaly Kanalı) ---
Kanal: E-2
Train Ortalama (mu): -0.4069 | Train Std (sigma): 0.5518
Test Zaman Adımı: 8532
Z-score > 3.0 ile Anomali İşaretlenen Adım Sayısı: 0

--- P-1 (Contextual Anomaly Kanalı) ---
Kanal: P-1
Train Ortalama (mu): -0.4272 | Train Std (sigma): 0.4028
Test Zaman Adımı: 8505
Z-score > 3.0 ile Anomali İşaretlenen Adım Sayısı: 124


In [8]:
def mask_to_sequences(mask: np.ndarray) -> list[list[int]]:
    """
    Boolean maskeyi ardışık [başlangıç, bitiş] indeks aralıklarına dönüştürür.
    Örnek: [0, 1, 1, 0, 1] -> [[1, 2], [4, 4]]
    """
    sequences = []
    in_sequence = False
    start_idx = 0
    
    for i, val in enumerate(mask):
        if val and not in_sequence:
            # Anomali başladı
            in_sequence = True
            start_idx = i
        elif not val and in_sequence:
            # Anomali bitti
            in_sequence = False
            sequences.append([start_idx, i - 1])
            
    # Eğer dizi anomali ile bitiyorsa sonlandır
    if in_sequence:
        sequences.append([start_idx, len(mask) - 1])
        
    return sequences

# P-1 için Z-score eşiğini 2.5 yapıp aralıkları çıkaralım
_, _, pred_p1_lower = run_zscore_baseline("P-1", threshold=2.5)
p1_predicted_seqs = mask_to_sequences(pred_p1_lower)

# E-2 için Z-score eşiğini 2.0 yapıp aralıkları çıkaralım
_, _, pred_e2_lower = run_zscore_baseline("E-2", threshold=2.0)
e2_predicted_seqs = mask_to_sequences(pred_e2_lower)

print("\n--- P-1 Tahmin Edilen Anomali Aralıkları (Eşik 2.5) ---")
print(f"Toplam Olay Sayısı: {len(p1_predicted_seqs)}")
print(f"İlk 5 Olay: {p1_predicted_seqs[:5]}")

print("\n--- E-2 Tahmin Edilen Anomali Aralıkları (Eşik 2.0) ---")
print(f"Toplam Olay Sayısı: {len(e2_predicted_seqs)}")
print(f"Tahmin Edilen Olaylar: {e2_predicted_seqs}")

Kanal: P-1
Train Ortalama (mu): -0.4272 | Train Std (sigma): 0.4028
Test Zaman Adımı: 8505
Z-score > 2.5 ile Anomali İşaretlenen Adım Sayısı: 277
Kanal: E-2
Train Ortalama (mu): -0.4069 | Train Std (sigma): 0.5518
Test Zaman Adımı: 8532
Z-score > 2.0 ile Anomali İşaretlenen Adım Sayısı: 179

--- P-1 Tahmin Edilen Anomali Aralıkları (Eşik 2.5) ---
Toplam Olay Sayısı: 66
İlk 5 Olay: [[10, 14], [111, 114], [257, 259], [462, 467], [512, 516]]

--- E-2 Tahmin Edilen Anomali Aralıkları (Eşik 2.0) ---
Toplam Olay Sayısı: 2
Tahmin Edilen Olaylar: [[5658, 5834], [5856, 5857]]


In [9]:
def check_overlap(seq_a: list[int], seq_b: list[int]) -> bool:
    """İki [başlangıç, bitiş] aralığının birbiriyle kesişip kesişmediğini denetler."""
    return max(seq_a[0], seq_b[0]) <= min(seq_a[1], seq_b[1])

def evaluate_channel_events(predicted_seqs: list[list[int]], ground_truth_seqs: list[list[int]]):
    """
    NASA JPL standardında Event-Level Precision, Recall ve False Alarm hesaplar.
    """
    # 1. Recall Denetimi: Hangi gerçek olaylar yakalandı?
    detected_gt = 0
    for gt in ground_truth_seqs:
        if any(check_overlap(gt, pred) for pred in predicted_seqs):
            detected_gt += 1
            
    recall = detected_gt / len(ground_truth_seqs) if ground_truth_seqs else 0.0
    
    # 2. Precision Denetimi: Tahminlerin kaç tanesi doğru, kaç tanesi sahte alarm?
    valid_predictions = 0
    false_alarms = 0
    for pred in predicted_seqs:
        if any(check_overlap(pred, gt) for gt in ground_truth_seqs):
            valid_predictions += 1
        else:
            false_alarms += 1
            
    precision = valid_predictions / len(predicted_seqs) if predicted_seqs else 0.0
    
    # F1-Score (Dengeleme Metriği)
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0.0
        
    return {
        "ground_truth_events": len(ground_truth_seqs),
        "predicted_events": len(predicted_seqs),
        "detected_events": detected_gt,
        "false_alarms": false_alarms,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

# P-1 ve E-2 için karne notlarını çıkaralım
gt_p1 = ast.literal_eval(df_labels[df_labels["chan_id"] == "P-1"]["anomaly_sequences"].iloc[0])
metrics_p1 = evaluate_channel_events(p1_predicted_seqs, gt_p1)

gt_e2 = ast.literal_eval(df_labels[df_labels["chan_id"] == "E-2"]["anomaly_sequences"].iloc[0])
metrics_e2 = evaluate_channel_events(e2_predicted_seqs, gt_e2)

print("--- P-1 (SMAP) Değerlendirme Karnesi ---")
for k, v in metrics_p1.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

print("\n--- E-2 (SMAP) Değerlendirme Karnesi ---")
for k, v in metrics_e2.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

--- P-1 (SMAP) Değerlendirme Karnesi ---
ground_truth_events: 3
predicted_events: 66
detected_events: 2
false_alarms: 62
precision: 0.0606
recall: 0.6667
f1_score: 0.1111

--- E-2 (SMAP) Değerlendirme Karnesi ---
ground_truth_events: 1
predicted_events: 2
detected_events: 1
false_alarms: 0
precision: 1.0000
recall: 1.0000
f1_score: 1.0000


In [10]:
# 82 Kanalın Tamamı Üzerinde Z-score Baseline Çalıştırma
baseline_results = []
DEFAULT_THRESHOLD = 3.0

print("82 kanal için Z-score baseline hesaplanıyor...")

for _, row in df_labels.iterrows():
    chan = row["chan_id"]
    sc = row["spacecraft"]
    gt_seqs = ast.literal_eval(row["anomaly_sequences"])
    
    # 1. Yalnızca Train'den ortalama ve standart sapma hesapla
    train_data = np.load(DATA_DIR / "train" / f"{chan}.npy")[:, 0]
    mu, sigma = np.mean(train_data), np.std(train_data)
    if sigma == 0:
        sigma = 1e-6
        
    # 2. Test Z-score
    test_data = np.load(DATA_DIR / "test" / f"{chan}.npy")[:, 0]
    z_scores = np.abs((test_data - mu) / sigma)
    pred_seqs = mask_to_sequences(z_scores > DEFAULT_THRESHOLD)
    
    # 3. Metrikleri hesapla
    metrics = evaluate_channel_events(pred_seqs, gt_seqs)
    
    baseline_results.append({
        "channel": chan,
        "spacecraft": sc,
        "gt_events": metrics["ground_truth_events"],
        "detected_events": metrics["detected_events"],
        "predicted_events": metrics["predicted_events"],
        "false_alarms": metrics["false_alarms"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"]
    })

# DataFrame'e dönüştür
df_results = pd.DataFrame(baseline_results)

# Şartname gereği reports/results.csv olarak kaydet
results_path = Path("../reports/results.csv")
df_results.to_csv(results_path, index=False)
print(f"Detaylı sonuçlar kaydedildi: {results_path}")

# --- GENEL ÖZET RAPORU ---
print("\n" + "="*50)
print("     NASA SPACEGUARD AI - BASELINE SONUÇ TABLOSU     ")
print("="*50)

for sc in ["SMAP", "MSL", "ALL"]:
    sub = df_results if sc == "ALL" else df_results[df_results["spacecraft"] == sc]
    total_gt = sub["gt_events"].sum()
    total_det = sub["detected_events"].sum()
    total_pred = sub["predicted_events"].sum()
    total_fa = sub["false_alarms"].sum()
    
    rec = total_det / total_gt if total_gt else 0.0
    prec = (total_pred - total_fa) / total_pred if total_pred else 0.0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) else 0.0
    
    print(f"\n[{sc}] Kanalları:")
    print(f"  Toplam Gerçek Olay (Ground Truth): {total_gt}")
    print(f"  Yakalanan Olay Sayısı: {total_det} (Recall: %{rec*100:.2f})")
    print(f"  Toplam Üretilen Alarm: {total_pred}")
    print(f"  Yanlış Alarm Sayısı (False Alarm): {total_fa} (Precision: %{prec*100:.2f})")
    print(f"  Event F1-Score: {f1:.4f}")

82 kanal için Z-score baseline hesaplanıyor...
Detaylı sonuçlar kaydedildi: ..\reports\results.csv

     NASA SPACEGUARD AI - BASELINE SONUÇ TABLOSU     

[SMAP] Kanalları:
  Toplam Gerçek Olay (Ground Truth): 69
  Yakalanan Olay Sayısı: 36 (Recall: %52.17)
  Toplam Üretilen Alarm: 1368
  Yanlış Alarm Sayısı (False Alarm): 998 (Precision: %27.05)
  Event F1-Score: 0.3563

[MSL] Kanalları:
  Toplam Gerçek Olay (Ground Truth): 36
  Yakalanan Olay Sayısı: 22 (Recall: %61.11)
  Toplam Üretilen Alarm: 217
  Yanlış Alarm Sayısı (False Alarm): 134 (Precision: %38.25)
  Event F1-Score: 0.4705

[ALL] Kanalları:
  Toplam Gerçek Olay (Ground Truth): 105
  Yakalanan Olay Sayısı: 58 (Recall: %55.24)
  Toplam Üretilen Alarm: 1585
  Yanlış Alarm Sayısı (False Alarm): 1132 (Precision: %28.58)
  Event F1-Score: 0.3767
